# 08 · Inter-rater reliability of the graded score

How much does a property's score depend on who inspects it? Four tests, on the reconciled
inspector identity from notebook 07 (`inspectors.py`):

1. **Consecutive inspections.** For each property's inspections in successive fiscal years,
   correlation and agreement when the same inspector returns versus when a different one
   does, holding the time between visits constant.
2. **Stability of inspector effects.** Fit property + inspector + year effects separately on
   two random halves of the properties. If inspector effects are real traits, an
   inspector's effect in one half predicts it in the other.
3. **Movers test.** Estimate inspector effects on one half of the properties, then ask, in the
   other half, whether a property's score change between two inspectors tracks the
   difference in their effects. A slope near 1 means inspector effects carry over to new
   properties at full size; near 0 means they reflect which properties inspectors are
   assigned.
4. **Near-simultaneous inspections.** Properties scored twice within 60 days. Code splits
   500+ unit complexes into several cases and inspects some properties in phases, so these
   pairs see the same property in the same condition.

Bootstrap intervals resample properties.

## Findings

Run 2026-10-05. Outputs are cleared before commit; the numbers below are from that run.

**The graded score is reproducible when the same inspector returns and barely reproducible
when a different one does. Inspector effects are stable, sizeable, and move with the
inspector.**

- **Consecutive inspections in different fiscal years.**

  | | Same inspector (524 pairs) | Different inspector (2,685 pairs) |
  |---|---|---|
  | Score correlation | 0.41 (95% 0.28–0.55) | 0.055 (0.02–0.09) |
  | Bottom-10 agreement (kappa) | 0.21 | 0.05 |
  | Same re-inspection band (1, 2 or 3 years) | 84% | 74% |

  The gap holds in every interval, including routine cycles over 1.75 years apart (0.40
  against 0.03), so timing and follow-up visits don't explain it.
- **Ordinary scores keep a rough order.** Rank correlations across different inspectors are
  0.33–0.37, so the order of ordinary scores partly survives a change of inspector. The low
  tail, where bottom-10 lives, does not.
- **Inspector effects are large.** Fitted with property and year effects, inspector effects
  have an SD of 4.9 points: −4.7 at the 10th percentile and +6.2 at the 90th, about an
  11-point spread. Year effects run from +3.6 (FY2020-21) to −3.7 (FY2022-23).
- **They are stable traits.** Fitted separately on random halves of the properties, an
  inspector's effect in one half correlates 0.84 with the other (median of 20 splits, range
  0.77–0.92).
- **They move with the inspector.** In the movers test, with effects estimated on one half
  and tested on the other, a property's score change between two inspectors tracks the
  predicted difference with slope 1.02 (95% 0.88–1.16; 2,994 pairs). Using only pairs with
  a different inspector, the slope is 1.05. The effect follows the inspector to new
  properties at full size.
- **The split-complex experiment isn't possible here.** 91 pairs of inspections of the same
  property fall within 60 days, but only 8 involve different inspectors.
- **Caveat.** Notebook 11 shows inspector effects also reflect where inspectors are assigned
  and how much they find, so a large effect isn't necessarily bias.

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd

import holdout as h
import reliability as rl
import temporal as t

pd.set_option("display.float_format", "{:.3f}".format)
insp = t.load_inspections()
insp = insp[insp.inspector >= 0]
print(f"{len(insp):,} scored inspections with a known inspector; {insp.address.nunique():,} properties; "
      f"{insp.inspector.nunique()} inspectors")


def band(score: pd.Series) -> pd.Series:
    # Code's re-inspection schedule: under 80 within 1 year, 80-90 within 2, 90+ within 3
    return pd.cut(score, [-1, 80, 90, 101], right=False, labels=["1 year", "2 years", "3 years"])

## 1. Consecutive inspections in different fiscal years

In [ ]:
pairs = rl.consecutive_pairs(insp)
pairs["gap"] = pd.cut(pairs.gap_years, [0, 1, 1.75, 10], labels=["under 1 year", "1-1.75 years", "over 1.75 years"])
pairs["band"], pairs["prev_band"] = band(pairs.score), band(pairs.prev_score)

rows = []
for (gap, same), d in pairs.groupby(["gap", "same_inspector"], observed=True):
    r, lo, hi = rl.cluster_bootstrap(d, lambda x: x.score.corr(x.prev_score), n=500)
    rows.append({
        "gap": gap, "inspector": "same" if same else "different", "pairs": len(d),
        "score r": r, "r 95% low": lo, "r 95% high": hi,
        "Spearman": d.score.corr(d.prev_score, method="spearman"),
        "bottom-10 kappa": rl.kappa(d.bottom10, d.prev_bottom10),
        "same schedule band": (d.band == d.prev_band).mean(),
    })
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
rows = []
for same, d in pairs.groupby("same_inspector"):
    r, lo, hi = rl.cluster_bootstrap(d, lambda x: x.score.corr(x.prev_score))
    rows.append({"inspector": "same" if same else "different", "pairs": len(d), "score r": r, "95% low": lo, "95% high": hi,
                 "bottom-10 kappa": rl.kappa(d.bottom10, d.prev_bottom10),
                 "same schedule band": (d.band == d.prev_band).mean(),
                 "median gap (years)": d.gap_years.median()})
print("all gaps pooled")
print(pd.DataFrame(rows).to_string(index=False))

## 2. Are inspector effects stable traits?

In [ ]:
full = rl.two_way_effects(insp)
n_by = insp.inspector.value_counts()
big = n_by[n_by >= 30].index
eff = full["rater"].reindex(big)
print(f"inspector effects, {len(big)} inspectors with 30+ inspections (points, relative to the average inspector):")
print(f"  SD {eff.std():.1f}; percentiles 10/50/90: {np.percentile(eff, [10, 50, 90]).round(1).tolist()}")
print("  year effects:", full["period"].round(1).to_dict())

rng = np.random.default_rng(h.SEED)
props = insp.address.unique()
split_r = []
for _ in range(20):
    half = set(rng.choice(props, size=len(props) // 2, replace=False))
    A, B = insp[insp.address.isin(half)], insp[~insp.address.isin(half)]
    eA, eB = rl.two_way_effects(A)["rater"], rl.two_way_effects(B)["rater"]
    keep = [k for k in big if (A.inspector == k).sum() >= 20 and (B.inspector == k).sum() >= 20]
    split_r.append(np.corrcoef(eA.reindex(keep), eB.reindex(keep))[0, 1])
print(f"split-half correlation of inspector effects, 20 random splits: median {np.median(split_r):.2f}, "
      f"range {min(split_r):.2f} to {max(split_r):.2f}")

## 3. Movers test

In [ ]:
rng = np.random.default_rng(h.SEED)
half = set(rng.choice(props, size=len(props) // 2, replace=False))
movers = []
for est_half, eval_half in [(True, False), (False, True)]:
    est = insp[insp.address.isin(half) == est_half]
    e = rl.two_way_effects(est)
    known = set(est.inspector.value_counts().loc[lambda s: s >= 20].index)
    p = pairs[pairs.address.isin(half) == eval_half]
    p = p[p.inspector.isin(known) & p.prev_inspector.isin(known)].copy()
    p["d_effect"] = (p.inspector.map(e["rater"]) - p.prev_inspector.map(e["rater"])
                     + p.fiscal_year.map(e["period"]) - p.prev_fiscal_year.map(e["period"]))
    p["d_score"] = p.score - p.prev_score
    movers.append(p)
movers = pd.concat(movers)


def slope(d: pd.DataFrame) -> float:
    x, y = d.d_effect.to_numpy(), d.d_score.to_numpy()
    return np.polyfit(x, y, 1)[0]


for label, d in [("all pairs", movers), ("different inspector only", movers[~movers.same_inspector])]:
    s, lo, hi = rl.cluster_bootstrap(d, slope)
    print(f"{label}: {len(d):,} pairs; slope of score change on predicted inspector + year change "
          f"{s:.2f} (95% {lo:.2f} to {hi:.2f}); correlation {d.d_score.corr(d.d_effect):.2f}")

## 4. Same property, inspected twice within 60 days

In [ ]:
near = rl.consecutive_pairs(insp, different_fy=False)
near = near[near.gap_years * 365.25 <= 60]
rows = []
for same, d in near.groupby("same_inspector"):
    rows.append({"inspector": "same" if same else "different", "pairs": len(d), "properties": d.address.nunique(),
                 "score r": d.score.corr(d.prev_score), "median |difference|": (d.score - d.prev_score).abs().median(),
                 "mean |difference|": (d.score - d.prev_score).abs().mean(),
                 "bottom-10 agreement": (d.bottom10 == d.prev_bottom10).mean(),
                 "median units": d.n_units.median()})
print(pd.DataFrame(rows).to_string(index=False))